In [ ]:
!pip install langchain langchain_core langchain_community langgraph langchain_google_genai langchain_huggingface

In [ ]:
!pip install faiss-cpu

In [ ]:
!pip install pypdf

In [ ]:
from typing import List, TypedDict, Literal
from pydantic import BaseModel, Field
import time
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings

from langgraph.graph import START, StateGraph, END

## Configuration
Set your API key using environment variables. Do NOT hardcode secrets in notebooks.

```bash
export GEMINI_API_KEY="your-api-key-here"
```

In [ ]:
# Get API key from environment variables
gemini_api_key = os.getenv("GEMINI_API_KEY")
if not gemini_api_key:
    raise ValueError("GEMINI_API_KEY environment variable is not set. Please set it before running this notebook.")

os.environ["GEMINI_API_KEY"] = gemini_api_key

## Load PDF Documents
Make sure the PDF files exist in the current directory: Company_Policies.pdf, Company_Profile.pdf, Product_and_Pricing.pdf

In [ ]:
# Load PDF documents with error handling
pdf_files = ["Company_Policies.pdf", "Company_Profile.pdf", "Product_and_Pricing.pdf"]
docs = []

for pdf_file in pdf_files:
    try:
        docs.extend(PyPDFLoader(pdf_file).load())
        print(f"✓ Successfully loaded {pdf_file}")
    except FileNotFoundError:
        print(f"✗ Warning: {pdf_file} not found. Skipping...")
    except Exception as e:
        print(f"✗ Error loading {pdf_file}: {e}")

if not docs:
    print("\n⚠️  No PDF files were loaded. Please ensure PDF files are present in the current directory.")
else:
    print(f"\n✓ Total documents loaded: {len(docs)}")

In [ ]:
# Split documents into chunks
if docs:
    chunks = RecursiveCharacterTextSplitter(
        chunk_size=600,
        chunk_overlap=100
    ).split_documents(docs)
    print(f"✓ Created {len(chunks)} text chunks")
else:
    chunks = []
    print("⚠️  No documents to chunk. Proceeding with empty chunk list.")

In [ ]:
# Initialize embeddings model
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("✓ Embeddings model loaded")

In [ ]:
# Create vector store
if chunks:
    vector_store = FAISS.from_documents(chunks, embeddings)
    retriever = vector_store.as_retriever(search_kwargs={"k": 4})
    print("✓ Vector store created and retriever initialized")
else:
    vector_store = None
    retriever = None
    print("⚠️  Vector store not created (no documents available)")

In [ ]:
# Initialize LLM
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
print("✓ LLM initialized")

In [ ]:
# Define State TypedDict with proper field types
class State(TypedDict):
    question: str
    needs_retrival: bool
    docs: List[Document]  # Changed from List[str] to List[Document]
    answer: str

In [ ]:
# Define retrieval decision model
class RetrieveDecision(BaseModel):
    should_retrieve: bool = Field(
        ..., description="True if external documents are needed to answer reliably. else False."
    )

decide_retrieval_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are an expert retrieval decision agent in a Retrieval-Augmented Generation (RAG) system.

Your task is to determine whether external knowledge retrieval is required before answering a user's question.

## Decision Rules

Set "should_retrieve" to true when:

1. The question requires factual information that may not be reliably contained in the model's parameters.
2. The user asks about recent, current, or time-sensitive events.
3. The question requires citations, evidence, references, statistics, research papers, or source-backed information.
4. The question is about specific companies, products, APIs, tools, documentation, repositories, laws, regulations, prices, or technical specifications.
5. The answer depends on proprietary, organization-specific, or rapidly changing information.
6. Additional context from external documents would improve accuracy.

Set "should_retrieve" to false when:

1. The question can be answered using general knowledge.
2. The request is for reasoning, brainstorming, summarization, rewriting, translation, or creative writing.
3. The task involves coding patterns, algorithms, mathematical reasoning, or conceptual explanations that do not require external facts.
4. The user is asking for opinions, recommendations, or general guidance that does not depend on specific documents.

## Output Requirements

Return only valid JSON."""
        ),
        ("human", "Question: {question}"),
    ]
)

In [ ]:
# Retrieval decision node
should_retrieve_llm = llm.with_structured_output(RetrieveDecision)

def decide_retrieval(state: State):
    """Decide whether to retrieve documents."""
    decision: RetrieveDecision = should_retrieve_llm.invoke(
        decide_retrieval_prompt.format_messages(question=state["question"])
    )
    return {"needs_retrival": decision.should_retrieve}

In [ ]:
# Direct answer generation prompt
direct_prompt_generation = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are an expert AI assistant.

Answer using only your internal knowledge.

Do not assume access to retrieved documents,
search results, tools, databases, or external sources.

If you can answer confidently, provide a clear and complete response.

If the answer depends on information you do not reliably know,
respond exactly:

"I don't know based on my general knowledge."

Never invent facts, citations, or sources.
"""
        ),
        ("human", "{question}")
    ]
)

In [ ]:
# Direct generation node
def generate_direct(state: State):
    """Generate answer using only internal knowledge."""
    out = llm.invoke(
        direct_prompt_generation.format_messages(question=state["question"])
    )
    return {
        "answer": out.content
    }

In [ ]:
# Retrieval node
def retrieve(state: State):
    """Retrieve relevant documents."""
    if retriever is None:
        print("⚠️  Warning: No retriever available (no documents loaded)")
        return {"docs": []}
    return {"docs": retriever.invoke(state["question"])}

In [ ]:
# Routing function
def route_after_decision(state: State) -> Literal["generate_direct", "retrieve"]:
    """Route to either direct generation or retrieval."""
    if state["needs_retrival"]:
        return "retrieve"
    return "generate_direct"

In [ ]:
# Build the graph
g = StateGraph(State)
g.add_node("decide_retrieval", decide_retrieval)
g.add_node("generate_direct", generate_direct)
g.add_node("retrieve", retrieve)

g.add_edge(START, "decide_retrieval")
g.add_conditional_edges("decide_retrieval", route_after_decision, ["retrieve", "generate_direct"])
g.add_edge("generate_direct", END)
g.add_edge("retrieve", END)

app = g.compile()
print("✓ Graph compiled successfully")
app

## Test Case 1: Company-Specific Question (requires retrieval)

In [ ]:
# Test with initial state properly initialized
result = app.invoke(
    {
        "question": "who is the ceo of Nexa AI",
        "needs_retrival": False,
        "docs": [],
        "answer": ""
    }
)

print("\n=== Test 1: CEO of Nexa AI ===")
print(f"Question: {result['question']}")
print(f"Needs Retrieval: {result['needs_retrival']}")
print(f"Answer: {result['answer'][:200]}..." if len(result['answer']) > 200 else f"Answer: {result['answer']}")

## Test Case 2: General Knowledge Question (no retrieval needed)

In [ ]:
result = app.invoke(
    {
        "question": "define agentic AI",
        "needs_retrival": False,
        "docs": [],
        "answer": ""
    }
)

print("\n=== Test 2: Define Agentic AI ===")
print(f"Question: {result['question']}")
print(f"Needs Retrieval: {result['needs_retrival']}")
print(f"Answer: {result['answer'][:300]}..." if len(result['answer']) > 300 else f"Answer: {result['answer']}")

## Test Case 3: Company Policy Question (will attempt retrieval)

In [ ]:
result = app.invoke({
    "question": "what is company policy"
})

print("\n=== Test 3: Company Policy ===")
print(f"Question: {result['question']}")
print(f"Needs Retrieval: {result['needs_retrival']}")
print(f"Number of Documents Retrieved: {len(result['docs'])}")
if result['docs']:
    print(f"\nFirst Document Preview: {result['docs'][0].page_content[:200]}...")
else:
    print("No documents retrieved")